# 02 - Premier tour brut, scénario par scénario

Ici on ne fait aucun redressage. On travaille sur un seul scénario du premier tour à la fois, sans mélanger les hypothèses concurrentes.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

try:
    from IPython.display import display
except ImportError:
    display = print

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
pio.renderers.default = "json"

from presidentielle2027.analytics.trends import smooth_candidate_trends
from presidentielle2027.dashboard.colors import get_political_color
from presidentielle2027.extraction.canonicalization import canonicalize_candidate_fields, is_generic_bloc_label

path = REPO_ROOT / "data" / "processed" / "wikipedia_2027_polls_normalized_v2.csv"
frame = pd.read_csv(path)
canonical = frame.apply(lambda row: canonicalize_candidate_fields(row.get("candidate_name"), row.get("candidate_party"), row.get("political_family")), axis=1, result_type="expand")
canonical.columns = ["candidate_name", "candidate_party", "political_family"]
frame[["candidate_name", "candidate_party", "political_family"]] = canonical
frame["publication_date"] = pd.to_datetime(frame["publication_date"], errors="coerce")
frame["estimate_percent"] = pd.to_numeric(frame["estimate_percent"], errors="coerce")
frame["sample_size"] = pd.to_numeric(frame.get("sample_size"), errors="coerce")
frame = frame.loc[(frame["round"] == "first_round") & (~frame["candidate_name"].map(is_generic_bloc_label))].copy()
scenario_catalog = frame.groupby("scenario_name", dropna=False).agg(rows=("poll_id", "count"), pollsters=("polling_company", "nunique"), candidates=("candidate_name", "nunique"), last_publication=("publication_date", "max")).sort_values(["rows", "last_publication"], ascending=[False, False])
display(scenario_catalog.head(30))

In [ ]:
SCENARIO_NAME = scenario_catalog.index[0]
scenario_frame = frame.loc[frame["scenario_name"] == SCENARIO_NAME].copy()
scenario_frame = smooth_candidate_trends(scenario_frame)

pd.Series({
    "scenario": SCENARIO_NAME,
    "rows": len(scenario_frame),
    "pollsters": scenario_frame["polling_company"].nunique(),
    "candidates": scenario_frame["candidate_name"].nunique(),
    "first_publication": scenario_frame["publication_date"].min(),
    "last_publication": scenario_frame["publication_date"].max(),
})

In [ ]:
latest_snapshot = scenario_frame.sort_values("publication_date", ascending=False).groupby("candidate_name", dropna=False).head(1)
latest_snapshot = latest_snapshot.sort_values("estimate_percent", ascending=False)
display(latest_snapshot[["candidate_name", "candidate_party", "polling_company", "publication_date", "estimate_percent", "sample_size", "source_url"]])

pollster_matrix = scenario_frame.pivot_table(index="candidate_name", columns="polling_company", values="estimate_percent", aggfunc="mean")
pollster_matrix

In [ ]:
figure = go.Figure()
for candidate_name, group in scenario_frame.groupby("candidate_name", dropna=False):
    ordered = group.sort_values("publication_date")
    party = ordered["candidate_party"].dropna().iloc[0] if ordered["candidate_party"].notna().any() else None
    family = ordered["political_family"].dropna().iloc[0] if ordered["political_family"].notna().any() else None
    color = get_political_color(party, family)
    figure.add_trace(go.Scatter(x=ordered["publication_date"], y=ordered["estimate_percent"], mode="markers", marker={"size": 7, "color": color, "opacity": 0.65}, name=f"{candidate_name} - points", legendgroup=candidate_name, showlegend=False))
    figure.add_trace(go.Scatter(x=ordered["publication_date"], y=ordered["smoothed_estimate"], mode="lines", line={"width": 2.5, "color": color}, name=candidate_name, legendgroup=candidate_name, showlegend=True))

figure.update_layout(title=f"Premier tour brut - {SCENARIO_NAME}", xaxis_title="Date de publication", yaxis_title="Intentions de vote (%)", paper_bgcolor="white", plot_bgcolor="white", legend_title="Candidats")
figure.update_xaxes(showgrid=True, gridcolor="#e5e5e5")
figure.update_yaxes(showgrid=True, gridcolor="#e5e5e5")
figure

In [ ]:
scenario_frame[["publication_date", "polling_company", "fieldwork_start_date", "fieldwork_end_date", "commissioner", "media_partner", "collection_method", "quota_method", "population", "candidate_name", "estimate_percent", "sample_size", "source_url"]].sort_values(["publication_date", "estimate_percent"], ascending=[False, False]).head(60)